In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import yaml
import logging
from datetime import datetime
import optuna
import xgboost as xgb
from optuna.visualization import plot_optimization_history
from sklearn.metrics import root_mean_squared_error
from data_preparation.data_processor import DataProcessor

# Suppress Optuna logging output
optuna.logging.get_logger("optuna").setLevel(logging.WARNING)

# Hyperparameter Tuning

For hyperparameter optimization, I used [Optuna](https://optuna.org/), an efficient framework that automates parameter search using Bayesian optimization and pruning techniques. The dataset was split with 80% reserved for training (65% for model training and 15% for validation during hyperparameter tuning) and 20% held out for testing.

## NAR Model

In [3]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}

lags = []

for waste in unique_waste:
    prep_data_company = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)
    waste_dfs[waste] = fetcher.create_xgboost_features(prep_data_company,waste_type= waste ,lags=lags, lagged_features= False,lagged_ratios= False ,fourier_terms= False, interaction_terms= False)

In [22]:
train_split_index = int(waste_dfs["Municipal"].shape[0] * 0.65)
val_split_index   = int(waste_dfs["Municipal"].shape[0] * 0.80)


# Store best params
best_params_all = {}

study_results = {}

for waste_type, df in waste_dfs.items():
    print(f"Tuning for {waste_type}...")

    # Time series split
    train_df = df.iloc[:train_split_index]
    val_df = df.iloc[train_split_index:val_split_index]

    X_train = train_df.drop(columns=["quantity_tons"])
    y_train = train_df["quantity_tons"]

    X_val = val_df.drop(columns=["quantity_tons"])
    y_val = val_df["quantity_tons"]

    def objective(trial):
        params = {
        "verbosity": 0,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "booster": "gbtree",
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
}


        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds)
        return rmse

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=60)

    best_params_all[waste_type] = study.best_params
    study_results[waste_type] = study
    print(f"Best RMSE for {waste_type}: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

# Save all params to YAML
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
yaml_file = f"best_params_NoLags_{timestamp}.yaml"
with open(yaml_file, 'w') as f:
    yaml.dump(best_params_all, f)

print(f"\nSaved all best parameters to: {yaml_file}")

for waste_type, study in study_results.items():
    fig = plot_optimization_history(study)
    fig.update_layout(title_text=f"Optuna Convergence - {waste_type}")
    fig.show()

Tuning for Municipal...
Best RMSE for Municipal: 42.2836
Best params: {'lambda': 9.956354853589941, 'alpha': 0.14245533461541682, 'colsample_bytree': 0.8561752723527161, 'subsample': 0.6010036512497885, 'learning_rate': 0.03368659550884364, 'n_estimators': 126, 'max_depth': 3, 'min_child_weight': 10}
Tuning for Industrial...
Best RMSE for Industrial: 42.9640
Best params: {'lambda': 0.005916324068842895, 'alpha': 0.03094857658963378, 'colsample_bytree': 0.8202995970328548, 'subsample': 0.9960604909483592, 'learning_rate': 0.02352848604170074, 'n_estimators': 138, 'max_depth': 3, 'min_child_weight': 8}
Tuning for Organic...
Best RMSE for Organic: 16.7991
Best params: {'lambda': 2.902005674715298, 'alpha': 0.17418090706689524, 'colsample_bytree': 0.6634298055670146, 'subsample': 0.6247638711286115, 'learning_rate': 0.011248716668398717, 'n_estimators': 625, 'max_depth': 5, 'min_child_weight': 8}
Tuning for Construction...
Best RMSE for Construction: 27.7440
Best params: {'lambda': 9.32859

## Lags

In [4]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}

lags = [6,7,13,14,20,21]

for waste in unique_waste:
    prep_data_company = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)
    waste_dfs[waste] = fetcher.create_xgboost_features(prep_data_company,waste_type= waste ,lags=lags, lagged_features= True,lagged_ratios= False ,fourier_terms= False, interaction_terms= False)

In [ ]:
train_split_index = int(waste_dfs["Municipal"].shape[0] * 0.65)
val_split_index   = int(waste_dfs["Municipal"].shape[0] * 0.80)


# Store best params
best_params_all = {}

study_results = {}

for waste_type, df in waste_dfs.items():
    print(f"Tuning for {waste_type}...")

    # Time series split
    train_df = df.iloc[:train_split_index]
    val_df = df.iloc[train_split_index:val_split_index]

    X_train = train_df.drop(columns=["quantity_tons"])
    y_train = train_df["quantity_tons"]

    X_val = val_df.drop(columns=["quantity_tons"])
    y_val = val_df["quantity_tons"]

    def objective(trial):
        params = {
        "verbosity": 0,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "booster": "gbtree",
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
}


        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds)
        return rmse

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=60)

    best_params_all[waste_type] = study.best_params
    study_results[waste_type] = study
    print(f"Best RMSE for {waste_type}: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

# Save all params to YAML
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
yaml_file = f"best_params_Lags_{timestamp}.yaml"
with open(yaml_file, 'w') as f:
    yaml.dump(best_params_all, f)

print(f"\nSaved all best parameters to: {yaml_file}")

for waste_type, study in study_results.items():
    fig = plot_optimization_history(study)
    fig.update_layout(title_text=f"Optuna Convergence - {waste_type}")
    fig.show()

Tuning for Municipal...
Best RMSE for Municipal: 43.3408
Best params: {'lambda': 1.3084776104797622, 'alpha': 0.4715264550541497, 'colsample_bytree': 0.5999192011944393, 'subsample': 0.6328964936856417, 'learning_rate': 0.06471045406782258, 'n_estimators': 152, 'max_depth': 3, 'min_child_weight': 4}
Tuning for Industrial...
Best RMSE for Industrial: 42.1999
Best params: {'lambda': 0.002821226977419814, 'alpha': 0.0014070041494897115, 'colsample_bytree': 0.5512793138175649, 'subsample': 0.9138564923661057, 'learning_rate': 0.03859244598594698, 'n_estimators': 101, 'max_depth': 7, 'min_child_weight': 8}
Tuning for Organic...
Best RMSE for Organic: 17.5678
Best params: {'lambda': 0.0020143599934621718, 'alpha': 4.395816420105612, 'colsample_bytree': 0.5232431639081302, 'subsample': 0.9288286180407345, 'learning_rate': 0.012468525853141094, 'n_estimators': 418, 'max_depth': 4, 'min_child_weight': 5}
Tuning for Construction...
Best RMSE for Construction: 27.9979
Best params: {'lambda': 0.06

## Lagged Ratios

In [5]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}

lags = [6,7,13,14,20,21]

for waste in unique_waste:
    prep_data_company = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)
    waste_dfs[waste] = fetcher.create_xgboost_features(prep_data_company,waste_type= waste ,lags=lags, lagged_features= False,lagged_ratios= True ,fourier_terms= False, interaction_terms= False)

In [ ]:
train_split_index = int(waste_dfs["Municipal"].shape[0] * 0.65)
val_split_index   = int(waste_dfs["Municipal"].shape[0] * 0.80)


# Store best params
best_params_all = {}

study_results = {}

for waste_type, df in waste_dfs.items():
    print(f"Tuning for {waste_type}...")

    # Time series split
    train_df = df.iloc[:train_split_index]
    val_df = df.iloc[train_split_index:val_split_index]

    X_train = train_df.drop(columns=["quantity_tons"])
    y_train = train_df["quantity_tons"]

    X_val = val_df.drop(columns=["quantity_tons"])
    y_val = val_df["quantity_tons"]

    def objective(trial):
        params = {
        "verbosity": 0,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "booster": "gbtree",
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
}


        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds)
        return rmse

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=60)

    best_params_all[waste_type] = study.best_params
    study_results[waste_type] = study
    print(f"Best RMSE for {waste_type}: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

# Save all params to YAML
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
yaml_file = f"best_params_LagRatios_{timestamp}.yaml"
with open(yaml_file, 'w') as f:
    yaml.dump(best_params_all, f)

print(f"\nSaved all best parameters to: {yaml_file}")

for waste_type, study in study_results.items():
    fig = plot_optimization_history(study)
    fig.update_layout(title_text=f"Optuna Convergence - {waste_type}")
    fig.show()

Tuning for Municipal...
Best RMSE for Municipal: 43.4791
Best params: {'lambda': 0.0020773182170332, 'alpha': 0.01683483639579107, 'colsample_bytree': 0.6582665744376233, 'subsample': 0.6960054441645495, 'learning_rate': 0.012795972404256976, 'n_estimators': 246, 'max_depth': 4, 'min_child_weight': 6}
Tuning for Industrial...
Best RMSE for Industrial: 42.3504
Best params: {'lambda': 0.0014328621040035697, 'alpha': 8.352383437525573, 'colsample_bytree': 0.5807840570971929, 'subsample': 0.9432868572744826, 'learning_rate': 0.015783955070317338, 'n_estimators': 112, 'max_depth': 4, 'min_child_weight': 10}
Tuning for Organic...
Best RMSE for Organic: 17.6353
Best params: {'lambda': 5.011072622067453, 'alpha': 0.07046662984501595, 'colsample_bytree': 0.5958990072850722, 'subsample': 0.9281087982411182, 'learning_rate': 0.010334605640054064, 'n_estimators': 639, 'max_depth': 8, 'min_child_weight': 1}
Tuning for Construction...
Best RMSE for Construction: 28.2721
Best params: {'lambda': 0.013